# Caso guiado Olist · De la teoría a la práctica (Sesiones 1 a 3)
**Maestría en Inteligencia Artificial y Data Science para la Transformación de Negocios · UMSA – Postgrado en Informática**
**Módulo:** Preparación, Recopilación y Análisis Exploratorio de Datos · **Docente:** M.Sc. Marcelo Pinto Macedo

---

### ¿Qué haremos?
Ya construimos la **base conceptual común** del módulo. Ahora la aplicamos, **sesión por sesión**, sobre un conjunto de datos sobre retrasos en vuelos en Estados Unidos — 2024 :

| Sección | Viene de | Qué aplicamos |
|---|---|---|
| **S1** | Sesión 1 · Introducción al análisis de datos | Tipos de datos y escalas de medición (Stevens, 1946) |
| **S2** | Sesión 2 · Recopilación de datos | Tamaño de muestra (Cochran, 1977) y muestreo estratificado |
| **S3** | Sesión 3 · Fuentes y extracción | Integración de tablas relacionadas por claves (Codd, 1970) |
| **Cierre** | CRISP-DM, fase 2 | Informe de recolección inicial de datos (Chapman et al., 2000) |

### La pregunta de negocio que guía la práctica
> **¿Que porcentaje de vuelos llegaron tarde?**

### Cómo trabajar
- Ejecuta las celdas **en orden** con **Mayús + Enter**.
- En las celdas **✍️ Tu turno**, completa el código donde dice `___`. Hay un ejercicio **básico** y, a veces, un **reto** opcional.
- Si algo falla, escribe tu duda en el chat.

## Paso 0 · El conjunto de datos y cómo lo obtenemos
**Airline Flight Operations and Delays Dataset** Es una muestra de datos reales y oficiales de la aviación comercial de Estados Unidos (extraída de los registros del Bureau of Transportation Statistics y optimizada en plataformas como GitHub/Kaggle), que recopila información operativa sobre vuelos, programación, rendimiento y factores de retraso..

- **Fuente oficial:** Kaggle (`kaggle.com/datasets/hrishitpatil/flight-data-2024`) y diccionario de datos (`github.com/Abdo-Anwar/flight-analytics-realtime-batch/blob/main/docs/data_dictionary.md`).
- **Formato:** CSV.
- **Acceso:** leemos cada archivo directamente desde su **dirección en GitHub** con `pd.read_csv(url)`. No hace falta descargarlo.

Usaremos cuatro tablas: **vuelos**, **retrasos_detalle**, **aeropuertos** y **aerolineas**. La carga puede tardar unos segundos.


In [129]:
import math
import pandas as pd

BASE = r"data/"
df = pd.read_csv(BASE +"flight_data_2024_sample.csv")

print("Vuelos: ", df.shape)

# adicion columna llave principal
df.insert(0,'id_flight', range(1,len(df)+1))

# Tabla de vuelos
vuelos = df[[
    'id_flight', 'year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
    'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'dest',
    'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off',
    'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay',
    'cancelled', 'cancellation_code', 'diverted',
    'crs_elapsed_time', 'actual_elapsed_time', 'air_time', 'distance'
]].rename(columns={'op_unique_carrier': 'id_carrier', 'origin': 'id_origin_airport', 'dest': 'id_dest_airport'}).copy()

# Tabla de retrasos_detalle
retrasos_detalle = df[[
    'id_flight', 'carrier_delay', 'weather_delay', 
    'nas_delay', 'security_delay', 'late_aircraft_delay'
]].copy()

# Aeropuertos (unifica origen y destino sin duplicados)
origenes = df[['origin', 'origin_city_name', 'origin_state_nm']].rename(
    columns={'origin': 'id_airport', 'origin_city_name': 'city_name', 'origin_state_nm': 'state_nm'}
)
destinos = df[['dest', 'dest_city_name', 'dest_state_nm']].rename(
    columns={'dest': 'id_airport', 'dest_city_name': 'city_name', 'dest_state_nm': 'state_nm'}
)
aeropuertos = pd.concat([origenes, destinos]).drop_duplicates(subset=['id_airport']).sort_values('id_airport').reset_index(drop=True)

# Aerolíneas (códigos IATA y nombres oficiales de EE. UU.)
nombres_aerolineas = {
    '9E': 'Endeavor Air', 'AA': 'American Airlines', 'AS': 'Alaska Airlines',
    'B6': 'JetBlue Airways', 'DL': 'Delta Air Lines', 'F9': 'Frontier Airlines',
    'G4': 'Allegiant Air', 'HA': 'Hawaiian Airlines', 'MQ': 'Envoy Air',
    'NK': 'Spirit Airlines', 'OH': 'PSA Airlines', 'OO': 'SkyWest Airlines',
    'QX': 'Horizon Air', 'UA': 'United Airlines', 'WN': 'Southwest Airlines',
    'YV': 'Mesa Airlines', 'YX': 'Republic Airways'
}

aerolineas = df[['op_unique_carrier']].drop_duplicates().rename(columns={'op_unique_carrier': 'id_carrier'}).sort_values('id_carrier').reset_index(drop=True)
aerolineas['carrier_name'] = aerolineas['id_carrier'].map(nombres_aerolineas).fillna('Otra Aerolínea')

print("1. Vuelos (Hechos):            ", vuelos.shape)
print("2. Retrasos detalle:           ", retrasos_detalle.shape)
print("3. Aeropuertos (Catálogo):     ", aeropuertos.shape)
print("4. Aerolíneas (Catálogo):      ", aerolineas.shape)



Vuelos:  (10000, 35)
1. Vuelos (Hechos):             (10000, 27)
2. Retrasos detalle:            (10000, 6)
3. Aeropuertos (Catálogo):      (312, 3)
4. Aerolíneas (Catálogo):       (15, 2)


**Resultado esperado:** vuelos (10000, 35) · Retrasos (10000, 8) · Rendimiento (10000, 14) · Horarios y Programación (10000, 18).

Observa que en las 3 tablas se tiene **misma cantidad de registros**.

---
# S1 · De la Sesión 1: tipos de datos y escalas de medición
**Recordemos:** Stevens (1946) propuso cuatro escalas — **nominal, ordinal, intervalo y razón** — y la escala determina qué cálculos son válidos.

### S1.1 · Primera mirada a horarios_y_programacion
`head()` muestra las primeras filas e `info()` el tipo de dato de cada columna.

In [130]:
vuelos.head(5)

,id_flight,year,month,day_of_month,day_of_week,fl_date,id_carrier,op_carrier_fl_num,id_origin_airport,id_dest_airport,...,crs_arr_time,arr_time,arr_delay,cancelled,cancellation_code,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance
0,1,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,RAP,...,1149,1139.0,-10.0,0,NaN,0,151.0,144.0,119.0,835.0
1,2,2024,1,1,1,2024-01-01,AA,148.0,CLT,PHX,...,1923,1906.0,-17.0,0,NaN,0,286.0,273.0,253.0,1773.0
2,3,2024,12,12,4,2024-12-12,9E,5440.0,CHA,ATL,...,1059,1042.0,-17.0,0,NaN,0,59.0,50.0,29.0,106.0
3,4,2024,4,8,1,2024-04-08,WN,1971.0,OMA,LAS,...,1430,1431.0,1.0,0,NaN,0,180.0,177.0,163.0,1099.0
4,5,2024,2,16,5,2024-02-16,WN,862.0,BWI,MYR,...,1510,1509.0,-1.0,0,NaN,0,90.0,96.0,76.0,399.0


In [131]:
pd.set_option("display.max_rows", None)
vuelos.info()

vuelos.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_flight            10000 non-null  int64  
 1   year                 10000 non-null  int64  
 2   month                10000 non-null  int64  
 3   day_of_month         10000 non-null  int64  
 4   day_of_week          10000 non-null  int64  
 5   fl_date              10000 non-null  str    
 6   id_carrier           10000 non-null  str    
 7   op_carrier_fl_num    10000 non-null  float64
 8   id_origin_airport    10000 non-null  str    
 9   id_dest_airport      10000 non-null  str    
 10  crs_dep_time         10000 non-null  int64  
 11  dep_time             9884 non-null   float64
 12  dep_delay            9884 non-null   float64
 13  taxi_out             9880 non-null   float64
 14  wheels_off           9880 non-null   float64
 15  wheels_on            9873 non-null   float64
 16

,id_flight,year,month,day_of_month,day_of_week,fl_date,id_carrier,op_carrier_fl_num,id_origin_airport,id_dest_airport,...,crs_arr_time,arr_time,arr_delay,cancelled,cancellation_code,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance
0,1,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,RAP,...,1149,1139.0,-10.0,0,NaN,0,151.0,144.0,119.0,835.0
1,2,2024,1,1,1,2024-01-01,AA,148.0,CLT,PHX,...,1923,1906.0,-17.0,0,NaN,0,286.0,273.0,253.0,1773.0
2,3,2024,12,12,4,2024-12-12,9E,5440.0,CHA,ATL,...,1059,1042.0,-17.0,0,NaN,0,59.0,50.0,29.0,106.0
3,4,2024,4,8,1,2024-04-08,WN,1971.0,OMA,LAS,...,1430,1431.0,1.0,0,NaN,0,180.0,177.0,163.0,1099.0
4,5,2024,2,16,5,2024-02-16,WN,862.0,BWI,MYR,...,1510,1509.0,-1.0,0,NaN,0,90.0,96.0,76.0,399.0


In [132]:
retrasos_detalle.info()

retrasos_detalle.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_flight            10000 non-null  int64
 1   carrier_delay        10000 non-null  int64
 2   weather_delay        10000 non-null  int64
 3   nas_delay            10000 non-null  int64
 4   security_delay       10000 non-null  int64
 5   late_aircraft_delay  10000 non-null  int64
dtypes: int64(6)
memory usage: 468.9 KB


,id_flight,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,0,0,0,0,0
1,2,0,0,0,0,0
2,3,0,0,0,0,0
3,4,0,0,0,0,0
4,5,0,0,0,0,0


In [133]:

aeropuertos.info()

aeropuertos.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_airport  312 non-null    str  
 1   city_name   312 non-null    str  
 2   state_nm    312 non-null    str  
dtypes: str(3)
memory usage: 7.4 KB


,id_airport,city_name,state_nm
0,ABE,"Allentown/Bethlehem/Easton, PA",Pennsylvania
1,ABI,"Abilene, TX",Texas
2,ABQ,"Albuquerque, NM",New Mexico
3,ABR,"Aberdeen, SD",South Dakota
4,ACK,"Nantucket, MA",Massachusetts


In [134]:
aerolineas.info()

aerolineas.head()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_carrier    15 non-null     str  
 1   carrier_name  15 non-null     str  
dtypes: str(2)
memory usage: 372.0 bytes


,id_carrier,carrier_name
0,9E,Endeavor Air
1,AA,American Airlines
2,AS,Alaska Airlines
3,B6,JetBlue Airways
4,DL,Delta Air Lines


**Qué observar:** las una columna de fechas aparecen como **texto** (`object` o `str`, según la versión de pandas) en la tabla **fl_date              **, no como fechas. Además, existen columnas `crs_dep_time`,`crs_arr_time` que estan como Int y no se encuentra en el formato time.

Mientras una fecha sea texto, no podemos restar fechas ni calcular retrasos. **Convertimos** las columnas con `pd.to_datetime`:

In [135]:
columnas_fecha = ["fl_date"]
for col in columnas_fecha:
    vuelos[col] = pd.to_datetime(vuelos[col])

print(vuelos[columnas_fecha].dtypes)
print("Primer vuelo del año:", vuelos["fl_date"].min())
print("Último vuelo del año:", vuelos["fl_date"].max())

fl_date    datetime64[us]
dtype: object
Primer vuelo del año: 2024-01-01 00:00:00
Último vuelo del año: 2024-12-31 00:00:00


In [136]:
columnas_hora = ["crs_dep_time","crs_arr_time"]
for col in columnas_hora:
    vuelos[col] = pd.to_datetime(vuelos[col].astype(str).str.zfill(4),
    format='%H%M',
    errors='coerce').dt.time
    
vuelos[["crs_dep_time","crs_arr_time"]].head(3)

,crs_dep_time,crs_arr_time
0,10:18:00,11:49:00
1,16:37:00,19:23:00
2,10:00:00,10:59:00


**Resultado esperado:** las columnas pasan a `datetime64`; el primer vuelo es del **01 de enero de 2024** y el último del **31 de diciembre de 2024**.

### S1.2 · Un número que no es un número
En la tabla de vuelos, el numero de vuelo **op_carrier_fl_num** se leyó como número flotante. Mira las primeras filas:

In [137]:
# Mira cómo el número de vuelo se leyó como decimal (float64)
vuelos[["op_carrier_fl_num", "id_carrier", "day_of_week"]].head(3)


,op_carrier_fl_num,id_carrier,day_of_week
0,3535.0,MQ,4
1,148.0,AA,1
2,5440.0,9E,4


**Qué observar:** el número de vuelo (`op_carrier_fl_num`) se leyó inicialmente con decimales (`float64`). El número de vuelo es una variable **nominal**: identifica una ruta o servicio comercial, no mide una cantidad. Como vimos en la Sesión 1 con los códigos de identificación, **sumarlo o promediarlo no tiene sentido**, por lo que debe guardarse y tratarse como **texto**.


In [138]:
# Lo convertimos a texto entero (nominal) porque promediar números de vuelo no tiene sentido
vuelos["op_carrier_fl_num"] = vuelos["op_carrier_fl_num"].astype(int).astype(str)
vuelos[["op_carrier_fl_num"]].head(3)


,op_carrier_fl_num
0,3535
1,148
2,5440


### S1.3 · Escala de cada variable
| Variable | Tabla | Escala | Cálculos válidos |
|---|---|---|---|
| `op_unique_carrier`, `origin`, `cancellation_code` | vuelos / catálogos | Nominal | Frecuencias, moda |
| `day_of_week` (1 = Lunes a 7 = Domingo) | vuelos | Ordinal | Mediana, percentiles |
| `fl_date` (fecha del vuelo) | vuelos | intervalo | Diferencia entre fechas |
| `dep_delay`, `arr_delay` (minutos de retraso) | vuelos | Intervalo | Mediana, medianda,desviación |
| `distance`, `air_time, weather_delay` | vuelos/retrasos | Razón | Todos |

Una variable **nominal** se resume con frecuencias:

In [139]:
# Variable nominal: frecuencia de vuelos por aerolínea
vuelos["id_carrier"].value_counts()

id_carrier
WN    2022
DL    1459
AA    1385
OO    1080
UA    1033
YX     434
MQ     406
NK     361
B6     331
OH     326
AS     323
F9     295
9E     278
G4     172
HA      95
Name: count, dtype: int64

In [140]:
# Variables de razón: estadísticos de distancia y tiempo en el aire
vuelos[["distance", "air_time", "arr_delay"]].describe().round(2)

,distance,air_time,arr_delay
count,10000.00,9836.00,9836.00
mean,838.35,115.45,7.55
std,598.53,70.74,55.80
min,31.00,8.00,-78.00
25%,404.00,64.00,-15.00
50%,680.00,98.00,-6.00
75%,1072.00,145.00,10.00
max,5095.00,635.00,2014.00


> ### ✍️ Tu turno · S1 · básico
> Muestra cantidad de vuelos por dia de la semana (1 a 7) ordenados.
>
> **Pista:** usa `value_counts()` y luego `sort_index()`.
> **Resultado esperado:** 1 → 1556 · 2 → 1400 · 3 → 1321 · 4 → 1552 · 5 → 1461 · 6 → 1300 · 5 → 1410

In [141]:
vuelos["day_of_week"].value_counts().sort_index()

day_of_week
1    1556
2    1400
3    1321
4    1552
5    1461
6    1300
7    1410
Name: count, dtype: int64

> ### ✍️ Tu turno · S1 · reto (opcional)
> Calcula la **media** y la **mediana** de los minutos de retraso en llegada `arr_delay`. ¿Cuál medida de tendencia representa mejor el comportamiento de un vuelo según la escala de Stevens y por qué?
>
> **Pista:** `mean()` y `median()`. Recuerda que `arr_delay` es una variable cuantitativa continua (intervalo/razón).

> **Resultado esperado:** media 7,55 min · mediana -6,0 min. Para una escala ordinal, lo riguroso es la **mediana** (y la distribución de frecuencias).

In [142]:
print("Media:  ", vuelos["arr_delay"].mean().round(2))
print("Mediana:", vuelos["arr_delay"].median())

Media:   7.55
Mediana: -6.0


---
# S2 · De la Sesión 2: tamaño de muestra y muestreo
**Recordemos:** la fórmula de **Cochran (1977)** con corrección por población finita, y la ventaja del **muestreo estratificado** para representar bien a los grupos pequeños.

### S2.1 · ¿Cuántas vuelos habría que revisar?
Supongamos que se desea auditar a mano una muestra de registros operativos de vuelos para clasificar las causas principales de retraso. La población de la muestra es de **10 000 vuelos**.

In [143]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2               # muestra para población muy grande
    return math.ceil(n0 / (1 + (n0 - 1) / N))    # corrección por población finita

n = tamano_muestra(len(vuelos))
n

370

**Resultado esperado:** **370** reseñas (95 % de confianza y 5 % de error).

### S2.2 · Aleatoria simple frente a estratificada
Los puntajes **no** están equilibrados: las reseñas de **2 estrellas** son solo el **3,2 %**. Comparamos cómo las representa cada muestra:

In [144]:
aleatoria = vuelos.sample(n=n, random_state=42)
estratificada = vuelos.groupby("cancelled", group_keys=False).sample(frac=n / len(vuelos), random_state=42)

comparacion = pd.DataFrame({
    "Población": vuelos["cancelled"].value_counts(normalize=True),
    "Aleatoria": aleatoria["cancelled"].value_counts(normalize=True),
    "Estratificada": estratificada["cancelled"].value_counts(normalize=True),
}).sort_index().round(3)
comparacion

,Población,Aleatoria,Estratificada
cancelled,,,
0,0.988,0.984,0.986
1,0.012,0.016,0.014


**Resultado esperado** para los vuelos cancelados: población **0,012** · aleatoria **0,016** · estratificada **0,014**.

**Conclusión:** la estratificada reproduce la población; la aleatoria subrepresentó a los vuelos cancelados, que justamente son las quejas que más interesa leer.

> ### ✍️ Tu turno · S2 · básico
> ¿Cuántas reseñas habría que revisar con un margen de error del **3 %**?
>
> **Pista:** usa la función `tamano_muestra` con el argumento `e=0.03`.
> **Resultado esperado:** 1 056 reseñas. Reducir el error del 5 % al 3 % casi triplica la muestra.

In [145]:
tamano_muestra(len(vuelos), e=0.03)

965

---
# S3 · De la Sesión 3: fuentes e integración de tablas
**Recordemos:** en una base de datos relacional (**Codd, 1970**) cada tabla guarda una entidad y las tablas se **relacionan por claves**.

### S3.1 · El modelo de datos de Flight Delay Dataset — 2024
```
retrasos_detalle ──(id_flight)── vuelos ──(id_carrier)── aerolineas
                                   │
                              (id_airport)
                                   │
                              aeropuertos

                            
```
- `id_flight` une **vuelos** con **retrasos_detalle**.
- `id_airport` une **aeropuertos** con **vuelos**.
- `id_carrier` une **aerolineas** con **vuelos**.

### S3.2 · ¿De qué estados de estados de Estados Unidos salen más vuelos?
El nombre de la ciudad y el estado del aeropuerto de origen están en la tabla `aeropuertos`. Las **unimos** con `merge`, equivalente a un `JOIN` en SQL:

In [146]:
vuelos_origen = vuelos.merge(aeropuertos, left_on="id_origin_airport", right_on="id_airport", how="left")
print(vuelos_origen.shape)
vuelos_origen["state_nm"].value_counts().head(5)

(10000, 30)


state_nm
California    1068
Texas         1067
Florida        941
New York       560
Illinois       539
Name: count, dtype: int64

**Resultado esperado:** (10000, 30) y, en primer lugar, **California** con 41 1068 vuelos, luego **Texas** con 1067 y **Florida** con 941.

### S3.3 · La pregunta de negocio: ¿Que porcentaje de vuelos llegaron tarde?
Según la aviación comercial (FAA/BTS), un vuelo se considera retrasado **"tarde"** si llega **15 minutos o más** después de su hora programada.(https://www.bts.gov/explore-topics-and-geography/topics/airline-time-performance-and-causes-flight-delays)

In [147]:
vuelos_operados = vuelos[vuelos["cancelled"] == 0].dropna(subset=["arr_delay"]).copy()
vuelos_operados["tarde"] = vuelos_operados["arr_delay"] >= 15

print("Vuelos operados analizados:", len(vuelos_operados))
print("Llegaron tarde (>=15 min):", vuelos_operados["tarde"].sum())
print("Porcentaje tarde:", round(vuelos_operados["tarde"].mean() * 100, 1), "%")

Vuelos operados analizados: 9836
Llegaron tarde (>=15 min): 2119
Porcentaje tarde: 21.5 %


**Resultado esperado:** 9836 vuelos analizados · 2119 retrasados · **21,5 %**.

> ### ✍️ Tu turno · S3 · básico
> Une **vuelos** con **aerolineas** usando la clave común `id_carrier` para incorporar el nombre oficial de cada compañía aérea.
>
> **Pista:** usa `on="id_carrier"` y `how="left"`.
> **Resultado esperado:** (10000, 28).

In [148]:
vuelos_aerolineas = vuelos.merge(aerolineas, on="id_carrier", how="left")
print(vuelos_aerolineas.shape)
vuelos_aerolineas[["id_flight", "id_carrier", "carrier_name", "arr_delay"]].head(5)

(10000, 28)


,id_flight,id_carrier,carrier_name,arr_delay
0,1,MQ,Envoy Air,-10.0
1,2,AA,American Airlines,-17.0
2,3,9E,Endeavor Air,-17.0
3,4,WN,Southwest Airlines,1.0
4,5,WN,Southwest Airlines,-1.0


> ### ✍️ Tu turno · S3 · reto (opcional)
> ¿Cuál es el porcentaje de vuelos que llegaron **tarde**  y el retraso promedio por aerolínea?
>
> **Pista:** une `vuelos_operados` con `aerolineas` y agrupar por `carrier_name`.
> **Resultado esperado:** una tabla ordenada de mayor a menor porcentaje de retraso, identificando a las aerolíneas con mayor índice de impuntualidad en 2024.

In [149]:
x = vuelos_operados.merge(aerolineas, on="id_carrier", how="left")

# Calculamos métricas por aerolínea usando funciones estándar
resumen_aerolineas = x.groupby("carrier_name").agg(
    total_vuelos=("id_flight", "count"),
    proporcion_tarde=("tarde", "mean"),
    retraso_medio_min=("arr_delay", "mean")
).round(2)

# Convertir proporción a porcentaje y ordenar
resumen_aerolineas["porcentaje_tarde"] = (resumen_aerolineas["proporcion_tarde"] * 100).round(1)
resumen_aerolineas[["total_vuelos", "porcentaje_tarde", "retraso_medio_min"]].sort_values(by="porcentaje_tarde", ascending=False)

,total_vuelos,porcentaje_tarde,retraso_medio_min
carrier_name,,,
Frontier Airlines,288,29.0,15.90
American Airlines,1360,28.0,16.20
Allegiant Air,170,28.0,9.73
JetBlue Airways,328,27.0,11.41
PSA Airlines,316,26.0,13.57
Spirit Airlines,350,24.0,8.47
United Airlines,1015,22.0,5.37
Southwest Airlines,1993,21.0,5.99
SkyWest Airlines,1067,20.0,9.12


---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)
La guía CRISP-DM (Chapman et al., 2000) pide documentar la tarea **«Recolectar los datos iniciales»**. Este es el informe del caso Flight Delay Dataset — 2024:

| Elemento | Caso Vuelos Comerciales Estados Unidos (2024) |
|---|---|
| **Fuente** | Bureau of Transportation Statistics (BTS)· repositorio oficial en GitHub y Kaggle(hrishitpatil/flight-data-2024) |
| **Tipo de fuente** | Externa · datos secundarios · registros operativos de aviación comercial |
| **Formato y acceso** | CSV · lectura directa (flight_data_2024_sample.csv) |
| **Tablas usadas** | vuelos (10 000) · retrasos_detalle (10 000) · aeropuertos (312) · aerolineas (15) |
| **Periodo** | 1 de enero de 2024 a 31 de diciembre de 2024 |
| **Problemas encontrados** | Fecha (fl_date) leídas como texto · horas programadas (crs_dep_time, crs_arr_time) como enteros HHMM · número de vuelo como decimal (float64) · valores nulos en retrasos y tiempos de aire correspondientes a los 122 vuelos cancelados y vuelos desviados |
| **Primer hallazgo** | De los **9 836** vuelos operados con registro válido de llegada, el **21,5 %** (**2 119** vuelos) experimentó un retraso significativo ($\ge 15$ min).|

### Para tu proyecto
Repite estas mismas secciones con **tu conjunto de datos** en la **plantilla espejo** (`Mi_proyecto_De_la_teoria_a_la_practica.ipynb`).

---
**Referencias:** Chapman, P. et al. (2000). *CRISP-DM 1.0*. SPSS. · Cochran, W. G. (1977). *Sampling techniques* (3.ª ed.). Wiley. · Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. · Olist (2018). *Brazilian E-Commerce Public Dataset by Olist*. Kaggle / GitHub. · Stevens, S. S. (1946). On the theory of scales of measurement. *Science*, 103(2684), 677–680.